**Capítulo 7 - Aprendizado Ensemble e Florestas Aleatórias**

*Este notebook mostra como combinar vários modelos para obter previsões melhores: votação, bagging, pasting, Florestas Aleatórias, importância de atributos, boosting, gradient boosting e stacking.*

*Este notebook contém a tradução e adaptação das células de exemplo do Capítulo 7. A seção final de soluções dos exercícios foi deixada fora do notebook principal e ficará organizada em `Respostas.md`.*

<table align="left">
  <td>
    <a href="https://colab.research.google.com/github/ageron/handson-ml3/blob/main/07_ensemble_learning_and_random_forests.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir no Colab"/></a>
  </td>
  <td>
    <a target="_blank" href="https://kaggle.com/kernels/welcome?src=https://github.com/ageron/handson-ml3/blob/main/07_ensemble_learning_and_random_forests.ipynb"><img src="https://kaggle.com/static/images/open-in-kaggle.svg" /></a>
  </td>
</table>

# Configuração

In [1]:
import sys
import sklearn
from packaging import version

assert sys.version_info >= (3, 7)
assert version.parse(sklearn.__version__) >= version.parse("1.0.1")

# Exercício 8 - Classificador por votação

Neste exercício, carregamos o MNIST, treinamos classificadores individuais e comparamos esses modelos com um `VotingClassifier`.

In [2]:
from sklearn.datasets import fetch_openml

mnist = fetch_openml('mnist_784',as_frame=False)

In [3]:
from sklearn.model_selection import train_test_split

X = mnist.data / 255.0
y = mnist.target.astype("uint8")

X_train_valid, X_test, y_train_valid, y_test = train_test_split(
    X,
    y,
    test_size=10_000,
    random_state=42,
    stratify=y
)

X_train, X_valid, y_train, y_valid = train_test_split(
    X_train_valid,
    y_train_valid,
    test_size=10_000,
    random_state=42,
    stratify=y_train_valid
)

X_train.shape, X_valid.shape, X_test.shape

((50000, 784), (10000, 784), (10000, 784))

In [4]:
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

random_forest_clf = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

extra_trees_clf = ExtraTreesClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

log_reg_clf = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=1_000, random_state=42, n_jobs=-1)
)

In [5]:
estimators = [
    ("Random Forest", random_forest_clf),
    ("Extra Trees", extra_trees_clf),
    ("Logistic Regression", log_reg_clf)
]

for name, estimator in estimators:
    print("Treinando:", name)
    estimator.fit(X_train, y_train)

Treinando: Random Forest
Treinando: Extra Trees
Treinando: Logistic Regression


In [6]:
from sklearn.metrics import accuracy_score

valid_scores = []

for name, estimator in estimators:
    y_pred = estimator.predict(X_valid)
    acc = accuracy_score(y_valid, y_pred)
    valid_scores.append((name, acc))
    print(f"{name}: {acc:.4f}")

Random Forest: 0.9710
Extra Trees: 0.9731
Logistic Regression: 0.9175


In [7]:
best_valid_name, best_valid_score = max(valid_scores, key=lambda score: score[1])

print("Melhor classificador individual na validação:", best_valid_name)
print(f"Acurácia de validação: {best_valid_score:.4f}")

Melhor classificador individual na validação: Extra Trees
Acurácia de validação: 0.9731


In [8]:
from sklearn.ensemble import VotingClassifier

voting_clf = VotingClassifier(
    estimators=[
        ("random_forest", random_forest_clf),
        ("extra_trees", extra_trees_clf),
        ("log_reg", log_reg_clf)
    ],
    voting="soft"
)

voting_clf.fit(X_train, y_train)

y_valid_pred = voting_clf.predict(X_valid)
voting_valid_acc = accuracy_score(y_valid, y_valid_pred)

print(f"VotingClassifier na validação: {voting_valid_acc:.4f}")

VotingClassifier na validação: 0.9539


In [9]:
y_test_pred = voting_clf.predict(X_test)
voting_test_acc = accuracy_score(y_test, y_test_pred)

print(f"VotingClassifier no teste: {voting_test_acc:.4f}")

VotingClassifier no teste: 0.9501


In [10]:
test_scores = []

for name, estimator in estimators + [("VotingClassifier", voting_clf)]:
    y_test_pred = estimator.predict(X_test)
    acc = accuracy_score(y_test, y_test_pred)
    test_scores.append((name, acc))
    print(f"{name}: {acc:.4f}")

Random Forest: 0.9657
Extra Trees: 0.9706
Logistic Regression: 0.9149
VotingClassifier: 0.9501


In [11]:
individual_test_scores = test_scores[:-1]
best_name, best_score = max(individual_test_scores, key=lambda score: score[1])

improvement = voting_test_acc - best_score

print("Melhor individual no teste:", best_name)
print(f"Acurácia do melhor individual: {best_score:.4f}")
print(f"Acurácia do VotingClassifier: {voting_test_acc:.4f}")
print(f"Melhoria: {improvement:.4f}")
print(f"Melhoria em pontos percentuais: {improvement * 100:.2f} p.p.")

Melhor individual no teste: Extra Trees
Acurácia do melhor individual: 0.9706
Acurácia do VotingClassifier: 0.9501
Melhoria: -0.0205
Melhoria em pontos percentuais: -2.05 p.p.


# Exercício 9 - Stacking

A partir daqui, reutilizamos os classificadores individuais treinados no exercício 8 e passamos para um novo exercício: criar um ensemble por stacking.

In [12]:
import numpy as np

X_valid_predictions = np.column_stack([
    estimator.predict(X_valid)
    for name, estimator in estimators
])

X_valid_predictions[:5]

array([[3, 3, 3],
       [3, 3, 3],
       [4, 4, 4],
       [0, 0, 0],
       [3, 3, 3]], dtype=uint8)

In [13]:
blender_clf = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

blender_clf.fit(X_valid_predictions, y_valid)

RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=42)

In [14]:
X_test_predictions = np.column_stack([
    estimator.predict(X_test)
    for name, estimator in estimators
])

stacking_manual_pred = blender_clf.predict(X_test_predictions)
stacking_manual_acc = accuracy_score(y_test, stacking_manual_pred)

print(f"VotingClassifier no teste: {voting_test_acc:.4f}")
print(f"Stacking manual no teste: {stacking_manual_acc:.4f}")
print(f"Diferença contra VotingClassifier: {(stacking_manual_acc - voting_test_acc) * 100:.2f} p.p.")

VotingClassifier no teste: 0.9501
Stacking manual no teste: 0.9679
Diferença contra VotingClassifier: 1.78 p.p.


In [15]:
from sklearn.ensemble import StackingClassifier

stacking_clf = StackingClassifier(
    estimators=[
        ("random_forest", random_forest_clf),
        ("extra_trees", extra_trees_clf),
        ("log_reg", log_reg_clf)
    ],
    final_estimator=RandomForestClassifier(
        n_estimators=200,
        random_state=42,
        n_jobs=-1
    ),
    cv="prefit",
    stack_method="predict_proba",
    n_jobs=-1
)

stacking_clf.fit(X_valid, y_valid)

StackingClassifier(cv='prefit',
                   estimators=[('random_forest',
                                RandomForestClassifier(n_jobs=-1,
                                                       random_state=42)),
                               ('extra_trees',
                                ExtraTreesClassifier(n_jobs=-1,
                                                     random_state=42)),
                               ('log_reg',
                                Pipeline(steps=[('standardscaler',
                                                 StandardScaler()),
                                                ('logisticregression',
                                                 LogisticRegression(max_iter=1000,
                                                                    n_jobs=-1,
                                                                    random_state=42))]))],
                   final_estimator=RandomForestClassifier(n_estimators=200,
                                                          n_jobs=-1,
                                                          random_state=42),
                   n_jobs=-1, stack_method='predict_proba')

In [16]:
stacking_clf_pred = stacking_clf.predict(X_test)
stacking_clf_acc = accuracy_score(y_test, stacking_clf_pred)

print(f"VotingClassifier no teste: {voting_test_acc:.4f}")
print(f"Stacking manual no teste: {stacking_manual_acc:.4f}")
print(f"StackingClassifier no teste: {stacking_clf_acc:.4f}")
print(f"Diferença StackingClassifier vs VotingClassifier: {(stacking_clf_acc - voting_test_acc) * 100:.2f} p.p.")
print(f"Diferença StackingClassifier vs stacking manual: {(stacking_clf_acc - stacking_manual_acc) * 100:.2f} p.p.")

VotingClassifier no teste: 0.9501
Stacking manual no teste: 0.9679
StackingClassifier no teste: 0.9720
Diferença StackingClassifier vs VotingClassifier: 2.19 p.p.
Diferença StackingClassifier vs stacking manual: 0.41 p.p.
